In [13]:
%reload_ext autoreload
%autoreload 2

In [14]:
# external imports
from pathlib import Path
import os
from dotenv import load_dotenv
from torch_geometric.data import Batch
import torch
import random

In [15]:
# internal imports
from primaite.network.generator import NetworkGenerator
from primaite.agents.aegis.modules.openai import OpenAIClient

Constants

In [16]:
NUMBER_OF_NODES = (10, 30) # lower and upper bounds for the number of nodes in the network
CORPUS_GRAPH_SIZES = [5, 10, 20, 40] # the graph sizes to generate
RANDOM_SEED = 165116 # random seed for reproducibility
TRAINING_CONFIG_PATH = Path("../agents") / "training_configs" / "do_nothing.yaml" # path to the training config file
SERVICE_NAMES = ["HTTP", "SSH", "FTP"] # list of service names to include in the network
PORTS_LIST = ["80", "22", "21"] # list of ports to include in the network
DATASET_SAVE_PATH = Path("../data") / "generated_data" / "data.pt" # path to save the generated dataset
DATASET_SIZE = 100 # number of networks to generate

Initialise OpenAI as the LLM

In [17]:
load_dotenv() 
open_ai_key = os.getenv('OPEN_AI_KEY')

openai = OpenAIClient(api_key=open_ai_key)

## Random Dataset Generation

This will produce laydowns, which are all saved in the same directory, and then we will use the laydowns to generate the datasets, which is saved as a pytorch file.


The sizes of the graphs is randomised between and upper and lower bound.

In [ ]:
outputs = []
for i in range(DATASET_SIZE):

    # save path for the laydown
    laydown_save_path = Path("../data") / "datasets" / "notebook_generated_dataset" / f"{i}.yaml"

    # intialise the generator
    generator = NetworkGenerator(graph_size=random.randint(NUMBER_OF_NODES[0], NUMBER_OF_NODES[1]), training_config_path=TRAINING_CONFIG_PATH, random_seed=1729, services=SERVICE_NAMES, ports=PORTS_LIST, pretrained_llm=openai)
    
    # run the generator
    output = generator.run_end_to_end(laydown_save_path=laydown_save_path)
    outputs.append(output)

    print(f"\nPROGRESS -------> {i + 1} of {DATASET_SIZE} datasets generated\n")

In [19]:
# example of the network generation
print(outputs[0])

Data(edge_index=[2, 36], id=[36], num_nodes=8, x=[8, 6], reasoning='The node SERVER_1 is the most critical node in the network as it is connected to multiple other nodes like computers, switches, and servers. If SERVER_1 is compromised, it can lead to a widespread attack and further spread to other nodes. Taking defensive action on SERVER_1 to ensure its security and prevent any potential attacks.', chosen_node='SERVER_1', node_property='HARDWARE', property_action='TURN_ON', service_name='NONE', laydown_filename='0.yaml')


In [ ]:
# save the outputs to a file
batch = Batch.from_data_list(outputs)
torch.save(batch, "../data/datasets/notebook_generated_dataset/dataset.pt")

## Diverse Network Generation

This code is used to create several datasets, each with a fixed, but different, number of nodes. The dataset size is divided by the number of graph sizes, so that each graph size will have a dataset of the same size.

For example, if the dataset size is 100 and the graph sizes are [5, 10, 20, 40], there are 4 different datasets (one for each graph size) and each will have 100/4 = 25 nodes.

In [ ]:
# save path for the generated datasets
laydown_root_dir = '../data/datasets/notebook_generated_dataset/'

# iterate through each of the corpus sizes
for corpus_graph_size in CORPUS_GRAPH_SIZES:

    # save the networks for each corpus size
    corpus = []    

    # create the directory if it doesn't exist
    if not os.path.exists(laydown_root_dir):
        os.mkdir(laydown_root_dir)

    # loop through as many networks as required
    for i in range(DATASET_SIZE // len(CORPUS_GRAPH_SIZES)):

        # change laydown root
        laydown_save_path = laydown_root_dir + f"{corpus_graph_size}_{i}.yaml"

        # init the generator
        generator = NetworkGenerator(graph_size=corpus_graph_size, training_config_path=TRAINING_CONFIG_PATH, random_seed=None, services=SERVICE_NAMES, ports=PORTS_LIST, pretrained_llm=openai)
        
        # create dataset item
        output = generator.run_end_to_end(laydown_save_path=laydown_save_path)
        corpus.append(output)

    # save each corpus
    batch = Batch.from_data_list(corpus)
    torch.save(batch, "../data/datasets/notebook_generated_dataset/dataset.pt")